-*- coding: utf-8 -*-
---
jupyter:
  jupytext:
    text_representation:
      extension: .jl
      format_name: percent
      format_version: '1.3'
      jupytext_version: 1.17.2
  kernelspec:
    display_name: Julia 1.11
    language: julia
    name: julia-1.11
---

# Trajectory-Seeded Traveling-Wave Discovery (Re=300)

This notebook uses a turbulent trajectory to generate “slow points” as
initial guesses for TW hookstep refinement. It complements random fuzzing
by focusing on dynamically relevant regions of state space.


In [31]:
using CloudAtlas
using LinearAlgebra
using Statistics
using Random
using Dates
using DelimitedFiles
using Serialization
using Base.Threads
using DifferentialEquations
using CairoMakie


## Configuration


In [42]:
Re = 300.0

# Domain sizes (α = 2π/Lx, γ = 2π/Lz)
α, γ = 2π/6.0, 2π/4.0

# Discretization
J, K, L = 2, 4, 7

# Symmetry group (combined generators)
sx, sy, sz, tx, tz = CloudAtlas.halfbox_symmetries()
H = [sz * tx]

# Trajectory settings
traj_tspan = (0.0, 500.0)
traj_saveat = 0.1
xnorm = 0.2
slow_window = 5
slow_vel_threshold = 0.5

# Hookstep parameters
hookparams = SearchParams(
    ftol = 1e-8,
    xtol = 1e-10,
    δ = 0.02,
    Nnewton = 20,
    Nhook = 4,
    Nmusearch = 6,
    verbosity = 0,
)

# Acceptance thresholds
norm_threshold = 1e-3
speed_threshold = 1e-5

# Output
out_dir = joinpath(@__DIR__, "tw_trajectory_seeded_re300")
mkpath(out_dir)


"/home/ebenq/Dev/julia/CloudAtlas.jl/notebooks/tw_discovery/tw_trajectory_seeded_re300"

## Helper types and functions


In [33]:
struct SolutionFingerprint
    cx::Float64
    cz::Float64
    nm::Float64
    shear::Float64
end

function fingerprint(model, ξ)
    x, cx, cz = extract_components(ξ, model)
    return SolutionFingerprint(cx, cz, norm(x), shear(x, model))
end

function is_distinct(new_fp::SolutionFingerprint, archive::Vector{SolutionFingerprint}; tol=(cx=1e-3, cz=1e-3, nm=2e-2, shear=2e-2))
    for fp in archive
        if isapprox(new_fp.cx, fp.cx, atol=tol.cx) &&
           isapprox(new_fp.cz, fp.cz, atol=tol.cz) &&
           isapprox(new_fp.nm, fp.nm, atol=tol.nm) &&
           isapprox(new_fp.shear, fp.shear, atol=tol.shear)
            return false
        end
    end
    return true
end

function random_guess(model, rng; xnorm=0.4)
    m = length(model)
    x = randn(rng, m)
    x = xnorm / norm(x) * x
    cx = randn(rng) * 0.1
    cz = randn(rng) * 0.1
    return [x; cx; cz]
end

function find_trajectory_minima(model, sol, D_matrix; window=5, vel_threshold=0.5)
    I_vals = [CloudAtlas.power_input(model, u) for u in sol.u]
    D_vals = [CloudAtlas.dissipation_rate(D_matrix, u) for u in sol.u]

    vel_id = zeros(length(sol.t))
    for i in 2:length(sol.t)
        dt = sol.t[i] - sol.t[i-1]
        dI = (I_vals[i] - I_vals[i-1]) / dt
        dD = (D_vals[i] - D_vals[i-1]) / dt
        vel_id[i] = sqrt(dI^2 + dD^2)
    end

    guesses = []
    for i in 1+window : length(vel_id)-window
        local_segment = vel_id[i-window : i+window]
        if vel_id[i] == minimum(local_segment) && vel_id[i] < vel_threshold
            x_guess = sol.u[i]
            cx_guess = model.keep_cx ? randn() * 0.01 : 0.0
            cz_guess = model.keep_cz ? randn() * 0.01 : 0.0
            ξ_guess = [x_guess; cx_guess; cz_guess]
            push!(guesses, (sol.t[i], ξ_guess))
        end
    end

    return guesses, I_vals, D_vals, vel_id
end


find_trajectory_minima (generic function with 1 method)

## Build model and generate trajectory


In [ ]:
model = ODEModel(α, γ, J, K, L, H; normalize=false, tw=true)


J,K,L,m == 2,4,7,338
(2J+1)(2K+1)(2L+1) + 1 == 676
Making matrices B,A1,A2,S3...
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 

retcode: Success
Interpolation: 1st order linear
t: 1001-element Vector{Float64}:
   0.0
   0.5
   1.0
   1.5
   2.0
   2.5
   3.0
   3.5
   4.0
   4.5
   ⋮
 496.0
 496.5
 497.0
 497.5
 498.0
 498.5
 499.0
 499.5
 500.0
u: 1001-element Vector{Vector{Float64}}:
 [-0.010253472900228957, 0.013763724353234849, -0.005301161307705585, -0.003649418454446727, 0.006200631772079462, -0.009708017798175981, -0.007445343524366803, 0.00467397099470344, -0.01396032534220859, 0.00421241359721908  …  0.012502783929400003, -0.016855194213139447, 0.01746846714905834, 0.006777332220281529, 0.007234403982977942, 0.0016114000212312282, -0.009434265436639472, -0.006691467967072053, 0.007151653267437594, -0.00039063439933113435]
 [-0.009536654264210638, 0.019328023828926556, -0.03254501485860494, -0.0015901767695169447, -0.007434825343223562, -0.01716667811118968, -0.04331361189356585, 0.002529214744407202, -0.01983577863709723, 0.00373037108976021  …  0.00730240844205467, -0.01697413548949006, 0.019287305991

In [43]:

# Integrate a turbulent trajectory (lab_frame=true gives drift in lab frame)
println("Integrating trajectory over t = $(traj_tspan)...")
seed_state = begin
    x0 = randn(length(model))
    x0 = xnorm / norm(x0) * x0
    ODEState(x0)
end
traj = integrate_flow(model, seed_state, traj_tspan; R=Re, saveat=traj_saveat, lab_frame=true)


Integrating trajectory over t = (0.0, 500.0)...


retcode: Success
Interpolation: 1st order linear
t: 5001-element Vector{Float64}:
   0.0
   0.1
   0.2
   0.3
   0.4
   0.5
   0.6
   0.7
   0.8
   0.9
   ⋮
 499.2
 499.3
 499.4
 499.5
 499.6
 499.7
 499.8
 499.9
 500.0
u: 5001-element Vector{Vector{Float64}}:
 [-0.017913270181091366, 0.0018368139629179967, 0.00618070967836504, 0.016733922642016345, 0.017963542769874365, 0.0040186972677061185, -0.020744877880109442, -0.006058200138422631, 0.0004727439972920567, -0.004202892482650979  …  0.011961380616636257, -0.007471431082824638, 0.006153202185689021, -0.021083759811990056, -0.009750024165887702, -0.007372749381155936, -0.0036382015052011034, 0.00046686777130974913, -0.009718285628189273, 0.01379554931736729]
 [-0.017659139541782342, 0.00030908496304154484, -0.0005105432735853809, 0.008714791789692613, 0.01071330843330388, -0.006103268157875194, -0.027328524248524, -0.004994421171317065, 0.000592102984184597, -0.0067725166449061625  …  0.012317312107550875, -0.00635074031235099, 0.005

## Plot trajectory + select slow points


In [44]:
D_matrix = build_dissipation_matrix(model)
guesses, I_vals, D_vals, vel_id = find_trajectory_minima(model, traj, D_matrix; window=slow_window, vel_threshold=slow_vel_threshold)
println("Selected $(length(guesses)) candidate slow points")

fig_traj = Figure(size=(800, 600))
ax = Axis(fig_traj[1, 1], xlabel="Dissipation (D)", ylabel="Power Input (I)", title="Trajectory in I-D plane")
lines!(ax, D_vals, I_vals, color=:black, alpha=0.6)
scatter!(ax, [D_vals[1]], [I_vals[1]], color=:green, marker=:circle)
scatter!(ax, [D_vals[end]], [I_vals[end]], color=:red, marker=:x)
CairoMakie.save(joinpath(out_dir, "trajectory_id_plane.png"), fig_traj)


Building dissipation matrix...
Selected 43 candidate slow points


## Refine slow points into TWs


In [46]:
solutions = Vector{Vector{Float64}}()
fp_archive = Vector{SolutionFingerprint}()
data_lock = ReentrantLock()

progress = Threads.Atomic{Int}(0)
progress_every = max(1, length(guesses) ÷ 100)

@threads for i in 1:length(guesses)
    _, ξ_guess = guesses[i]

    f(ξ) = model.g(ξ, Re)
    Df(ξ) = model.Dg(ξ, Re)

    ξ_star, converged = CloudAtlas.hookstepsolve(f, Df, ξ_guess, hookparams)

    if converged
        x, cx, cz = extract_components(ξ_star, model)
        println("Found converged TW guess at t=$(guesses[i][1]): ||x||=$(norm(x)), cx=$(cx), cz=$(cz)")
        if norm(x) > norm_threshold && (abs(cx) > speed_threshold || abs(cz) > speed_threshold)
            fp = fingerprint(model, ξ_star)
            lock(data_lock) do
                if is_distinct(fp, fp_archive)
                    push!(fp_archive, fp)
                    push!(solutions, ξ_star)
                end
            end
        end
    end

    done = Threads.atomic_add!(progress, 1)
    if done % progress_every == 0 || done == length(guesses)
        pct = round(100 * done / length(guesses); digits=1)
        println("Progress: $(done)/$(length(guesses)) ($(pct)%) (unique=$(length(solutions)))")
    end
end

println("Found $(length(solutions)) unique TWs from trajectory seeding.")


Progress: 0/43 (0.0%) (unique=0)
Progress: 1/43 (2.3%) (unique=0)
Progress: 2/43 (4.7%) (unique=0)
Progress: 3/43 (7.0%) (unique=0)
Progress: 4/43 (9.3%) (unique=0)
Progress: 5/43 (11.6%) (unique=0)
Progress: 6/43 (14.0%) (unique=0)
Progress: 7/43 (16.3%) (unique=0)
Progress: 8/43 (18.6%) (unique=0)
Progress: 9/43 (20.9%) (unique=0)
Progress: 10/43 (23.3%) (unique=0)
Progress: 11/43 (25.6%) (unique=0)
Progress: 12/43 (27.9%) (unique=0)
Progress: 13/43 (30.2%) (unique=0)
Progress: 14/43 (32.6%) (unique=0)
Progress: 15/43 (34.9%) (unique=0)
Progress: 16/43 (37.2%) (unique=0)
Progress: 17/43 (39.5%) (unique=0)
Found converged TW guess at t=454.4: ||x||=3.1434354272009615e-13, cx=8.710777947836747e-10, cz=0.001191896044162709
Progress: 18/43 (41.9%) (unique=0)
Found converged TW guess at t=456.3: ||x||=3.03945606424293e-13, cx=8.677854411401623e-10, cz=0.012920988080969292
Progress: 19/43 (44.2%) (unique=0)
Found converged TW guess at t=458.2: ||x||=2.934833830681467e-13, cx=8.644676333566

## Save results


In [37]:
summary_path = joinpath(out_dir, "solutions_summary.csv")
open(summary_path, "w") do io
    println(io, "id,cx,cz,norm,shear")
    for (i, ξ) in enumerate(solutions)
        x, cx, cz = extract_components(ξ, model)
        println(io, "$(i),$(cx),$(cz),$(norm(x)),$(shear(x, model))")
    end
end

serialized_path = joinpath(out_dir, "solutions.bin")
open(serialized_path, "w") do io
    serialize(io, solutions)
end


## Quick visualization


In [38]:
if !isempty(solutions)
    D_matrix = build_dissipation_matrix(model)
    I_vals = [power_input(model, extract_components(ξ, model)[1]) for ξ in solutions]
    D_vals = [dissipation_rate(D_matrix, extract_components(ξ, model)[1]) for ξ in solutions]

    fig = Figure(size=(700, 500))
    ax = Axis(fig[1, 1], xlabel="Dissipation (D)", ylabel="Power Input (I)", title="Trajectory-seeded TWs")
    scatter!(ax, D_vals, I_vals, color=:dodgerblue, markersize=10)
    save(joinpath(out_dir, "tw_seeded_id_plane.png"), fig)
end
